# 1 — Mathematical Foundations of Chemical Reservoir Computing

This notebook develops the mathematical framework that underpins the
`chemical_reservoir` package.  We cover:

1. Reservoir computing (general theory)
2. Chemical reaction networks & mass-action kinetics
3. The chemical reservoir as a dynamical system
4. Input encoding & readout via ridge regression
5. Feature expansion
6. Bayesian optimisation of reservoir parameters
7. Error metrics & evaluation

Mathematical notation is rendered inline throughout.

---
## 1.1  Reservoir Computing — General Framework

A **reservoir computer** maps an input signal $\mathbf{u}(t)$ to an output
$\mathbf{y}(t)$ through a high-dimensional, nonlinear dynamical system (the
*reservoir*) followed by a trained linear *readout*.

### State update

$$
\frac{d\mathbf{x}}{dt} = F\bigl(\mathbf{x}(t),\, \mathbf{u}(t)\bigr)
$$

where $\mathbf{x}(t) \in \mathbb{R}^N$ is the reservoir state and $F$ is a
fixed (untrained) nonlinear map.

### Readout

$$
\hat{\mathbf{y}}(t) = W_{\text{out}}\, \phi\bigl(\mathbf{x}(t)\bigr)
$$

where $\phi$ is an optional feature expansion and $W_{\text{out}}$ are the
only trained weights.

**Key properties**:
- *Echo-state property*: the reservoir state asymptotically forgets initial
  conditions.
- *Separation property*: distinct inputs produce distinct reservoir
  trajectories.
- *Approximation*: with enough reservoir nodes a linear readout can
  approximate any fading-memory functional.

---
## 1.2  Chemical Reaction Networks

A chemical reaction network (CRN) is a set of $M$ reactions among $N$ species
$S_1, \dots, S_N$.  Reaction $j$ has the form

$$
\sum_{i=1}^{N} \alpha_{ij}\, S_i \;\xrightarrow{k_j}\;
\sum_{i=1}^{N} \beta_{ij}\, S_i
$$

where $\alpha_{ij}$ and $\beta_{ij}$ are stoichiometric coefficients and
$k_j > 0$ is the rate constant.

### Mass-action kinetics

Under mass-action kinetics the rate of reaction $j$ is

$$
v_j(\mathbf{c}) = k_j \prod_{i=1}^{N} c_i^{\alpha_{ij}}
$$

and the concentration of species $i$ evolves as

$$
\frac{dc_i}{dt}
= \sum_{j=1}^{M} (\beta_{ij} - \alpha_{ij})\, v_j(\mathbf{c})
\;+\; r_i(t)
$$

where $r_i(t)$ is an external input rate for species $i$ (zero for
non-input species).

---
## 1.3  The Chemical Reservoir

We use the CRN as the reservoir $F$.  Concretely:

1. **Species concentrations** $\mathbf{c}(t) \in \mathbb{R}^N_{\geq 0}$
   serve as the reservoir state $\mathbf{x}$.
2. **Input channels** are a subset of species whose input rates
   $r_i(t)$ are modulated by the driving signal $\mathbf{u}(t)$.
3. **Output channels** are a (possibly different) subset of species
   whose concentrations are observed for the readout.

The full state equation becomes:

$$
\frac{dc_i}{dt}
= \underbrace{\sum_{j} (\beta_{ij} - \alpha_{ij})\, k_j \prod_{\ell} c_\ell^{\alpha_{\ell j}}}_{\text{reaction dynamics}}
\;+\;
\underbrace{\gamma_i\, u_{\pi(i)}(t)}_{\text{input encoding}}
$$

where $\gamma_i$ is the input gain and $\pi(i)$ maps species $i$ to an
input signal component.

### Why CRNs make good reservoirs

- Polynomial nonlinearities from mass-action kinetics.
- Natural dissipation (forward rates ≫ reverse rates) provides the echo-state property.
- Network topology controls the complexity of the dynamics.
- Physical constraints (non-negative concentrations) act as natural regularisers.

---
## 1.4  Input Encoding & Ridge-Regression Readout

### Encoding the target ODE/PDE

Given a target ODE

$$\frac{d\mathbf{y}}{dt} = g(\mathbf{y}, t)$$

with ground-truth solution $\mathbf{y}^*(t)$, we:

1. Solve the ODE numerically on a **training interval** $[0, T_{\text{train}}]$.
2. Use $\mathbf{y}^*(t)$ (or a derived signal) as the input $\mathbf{u}(t)$
   to the reservoir.
3. Simulate the reservoir forward, recording the output-channel
   concentrations.

### Feature matrix

Let $\mathbf{c}_{\text{out}}(t_k) \in \mathbb{R}^p$ be the output-channel
concentrations at time $t_k$.  With **quadratic feature expansion**:

$$
\phi(\mathbf{c}) = \bigl[1,\; c_1, \dots, c_p,\;
c_1^2, c_1 c_2, \dots, c_p^2\bigr]^\top
$$

The feature matrix is $\Phi \in \mathbb{R}^{n_{\text{train}} \times d}$
where $d = 1 + p + \binom{p+1}{2}$.

### Ridge regression

We learn $W_{\text{out}}$ by minimising

$$
\min_{W} \;\lVert \Phi\, W^\top - Y_{\text{train}} \rVert_F^2
\;+\; \lambda \lVert W \rVert_F^2
$$

where $Y_{\text{train}} \in \mathbb{R}^{n_{\text{train}} \times q}$ is the
target and $\lambda$ is the regularisation strength (chosen by
cross-validation via `RidgeCV`).

The closed-form solution is:

$$
W^\top = (\Phi^\top \Phi + \lambda I)^{-1} \Phi^\top Y_{\text{train}}
$$

---
## 1.5  Bayesian Optimisation

The reservoir's performance depends on continuous hyperparameters
$\boldsymbol{\theta}$ (input rates $\gamma_i$, initial concentrations
$c_i(0)$).  We tune them with Bayesian optimisation:

1. Model the objective $f(\boldsymbol{\theta}) = \text{RMSE}(\boldsymbol{\theta})$
   with a **Gaussian Process** (GP) surrogate:

$$
f(\boldsymbol{\theta}) \sim \mathcal{GP}\bigl(m(\boldsymbol{\theta}),\;
k(\boldsymbol{\theta}, \boldsymbol{\theta}')\bigr)
$$

2. Select the next evaluation point by maximising an **acquisition function**:

| Acquisition | Formula |
|---|---|
| Expected Improvement (EI) | $\text{EI}(\boldsymbol{\theta}) = \mathbb{E}\bigl[\max(f_{\text{best}} - f(\boldsymbol{\theta}), 0)\bigr]$ |
| Probability of Improvement (PI) | $\text{PI}(\boldsymbol{\theta}) = \Phi\!\left(\frac{f_{\text{best}} - \mu(\boldsymbol{\theta})}{\sigma(\boldsymbol{\theta})}\right)$ |
| Upper Confidence Bound (UCB) | $\text{UCB}(\boldsymbol{\theta}) = \mu(\boldsymbol{\theta}) - \kappa\,\sigma(\boldsymbol{\theta})$ |

3. Evaluate $f$ at the selected point, update the GP, and repeat.

---
## 1.6  PDE Extension (Method of Lines)

For a 1-D PDE

$$
\frac{\partial u}{\partial t}
= \mathcal{L}\bigl(u, u_x, u_{xx}, x, t\bigr)
$$

on $[a, b] \times [0, T]$, we semi-discretise in space using $n_x$ grid
points, converting the PDE into a system of $n_x$ coupled ODEs:

$$
\frac{d\mathbf{U}}{dt} = \mathbf{G}(\mathbf{U}, t)
$$

where $\mathbf{U}(t) = [u(x_1, t), \dots, u(x_{n_x}, t)]^\top$ and
spatial derivatives are approximated by finite differences:

$$
\frac{\partial u}{\partial x}\bigg|_{x_i} \approx \frac{U_{i+1} - U_{i-1}}{2\Delta x},
\qquad
\frac{\partial^2 u}{\partial x^2}\bigg|_{x_i} \approx \frac{U_{i+1} - 2U_i + U_{i-1}}{\Delta x^2}
$$

The resulting ODE system is then treated identically to the ODE case above.

---
## 1.7  Error Metrics

| Metric | Formula | Interpretation |
|--------|---------|----------------|
| RMSE | $\sqrt{\frac{1}{n}\sum_k \lVert \hat{\mathbf{y}}_k - \mathbf{y}_k \rVert^2}$ | Scale-dependent error |
| MAE | $\frac{1}{n}\sum_k \lVert \hat{\mathbf{y}}_k - \mathbf{y}_k \rVert_1$ | Robust to outliers |
| NRMSE | $\frac{\text{RMSE}}{\text{range}(\mathbf{y})}$ | Normalised, comparable across problems |
| $R^2$ | $1 - \frac{\sum_k \lVert \hat{\mathbf{y}}_k - \mathbf{y}_k \rVert^2}{\sum_k \lVert \mathbf{y}_k - \bar{\mathbf{y}} \rVert^2}$ | Variance explained |
| Pearson $r$ | $\text{corr}(\hat{\mathbf{y}}, \mathbf{y})$ | Linear correlation |
| Error growth rate | $\frac{d}{dt}\log \lVert e(t) \rVert$ | Stability / Lyapunov-like divergence |

---
## 1.8  Demonstration — Putting It All Together

Below we build a minimal end-to-end example to illustrate the
mathematics in code.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import chemical_reservoir as cr

# Load a reaction network
from chemical_reservoir.networks import get_builtin_networks, get_network_io_config

networks = get_builtin_networks()
print(f"Available networks: {list(networks.keys())}")
print(f"\nSimple autocatalytic network has {len(networks['simple_autocatalytic'])} reactions:")
for rxn in networks['simple_autocatalytic']:
    print(f"  {rxn['name']}: {rxn['species']}  "
          f"{rxn['reactants']} -> {rxn['products']}  (k={rxn['rate']})")

In [ ]:
# Create a target ODE — the Van der Pol oscillator
problem = cr.problems.create_vanderpol_problem(mu=1.5)

# Solve the ground-truth ODE
t = np.linspace(*problem.t_span, 500)
y_true = problem.solve_numerical(t)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for i, name in enumerate(problem.variable_names):
    axes[0].plot(t, y_true[:, i], label=name)
axes[0].set(xlabel='t', ylabel='y', title='Van der Pol — Ground Truth')
axes[0].legend()

axes[1].plot(y_true[:, 0], y_true[:, 1])
axes[1].set(xlabel='x', ylabel='v', title='Phase Portrait')
plt.tight_layout()
plt.show()

In [ ]:
# Create reservoir from a high-dimensional Brusselator network
# (5 coupled oscillator modules, 20 species, 50 reactions)
reservoir, config = cr.io.load_reservoir_from_json(
    "../network_configs/high_dimensional_random.json"
)
# Use time-delay embedding for better generalisation
reservoir.feature_expansion = 'delayed'

print(f"Reservoir: {config['name']}")
print(f"  Species:   {reservoir.n_species}")
print(f"  Reactions: {reservoir.n_reactions}")
print(f"  Inputs:    {reservoir.input_channels}")
print(f"  Outputs:   {len(reservoir.output_channels)} channels")

results = reservoir.solve_ode(problem, n_train=1600, n_test=800)

metrics = cr.MetricsCalculator.calculate_all(
    y_true=results['y_true'],
    y_pred=results['y_pred'],
    t=results['t_all'],
    n_train=len(results['t_train']),
    variable_names=problem.variable_names,
)

print(f"\nFeatures:   {reservoir.training_info['n_features']}")
print(f"Ridge α:    {reservoir.training_info['ridge_alpha']:.6f}")
print(f"Train RMSE: {metrics.train_rmse:.4e}")
print(f"Test  RMSE: {metrics.test_rmse:.4e}")
print(f"Test  R²:   {metrics.test_r2:.4f}")

In [ ]:
# Visualise: overlay true vs predicted
n_train = len(results['t_train'])
t_all = results['t_all']

fig, axes = plt.subplots(len(problem.variable_names), 1,
                         figsize=(12, 3 * len(problem.variable_names)), sharex=True)
if len(problem.variable_names) == 1:
    axes = [axes]

for i, name in enumerate(problem.variable_names):
    ax = axes[i]
    ax.plot(t_all, results['y_true'][:, i], 'b-', label='True', alpha=0.8)
    ax.plot(t_all, results['y_pred'][:, i], 'r--', label='Predicted', alpha=0.8)
    ax.axvline(t_all[n_train - 1], color='k', ls=':', label='Train/Test boundary')
    ax.set_ylabel(name)
    ax.legend(loc='upper right')

axes[-1].set_xlabel('t')
fig.suptitle('Chemical Reservoir: True vs Predicted', y=1.02)
plt.tight_layout()
plt.show()

---
## Summary

This notebook covered the core mathematical building blocks:

- **Reservoir computing** maps inputs through a nonlinear dynamical system
  and trains only a linear readout.
- **Mass-action kinetics** provide the nonlinear dynamics via polynomial
  reaction rates.
- **Ridge regression** with optional quadratic features learns the readout
  weights.
- **Bayesian optimisation** tunes continuous reservoir hyperparameters.
- **Method of lines** extends the approach from ODEs to 1-D PDEs.
- **Open-loop vs closed-loop**: in open-loop mode, the encoding drives the
  reservoir for the full time domain; in closed-loop mode, predicted outputs
  are fed back to modulate the encoding during testing, enabling autonomous
  attractor tracking.  See `solve_ode_closed_loop()` and **Notebook 05** for
  a detailed comparison.

Continue to **Notebook 02** for a hands-on quick-start tutorial.